In [ ]:
"""
Episode start/end intention labeling using a weighted keyword dictionary (Subcat-style, iterative).

Reads (from your Windows paths by default):
  - Episodes CSV (All_episodes_with_messages_TrueFalse.csv)
  - Dictionary file (dictionary.xlsx OR dictionary.csv) in wide format:
      one column per label, one keyword per cell
    Optional weight columns per label:
      <LabelName>__weight  (numeric)

Optional:
  - A Blacklist column (case-insensitive name: "Blacklist") with optional "Blacklist__weight".
    Blacklist matches do NOT become a label; they are treated as "noise" indicators.

Writes (to out_dir):
  - episodes_with_intentions.csv
  - start_intent_candidate_keywords.csv
  - end_intent_candidate_keywords.csv

Key improvements:
  - Keeps digits in tokens (e2e, v2, aab…)
  - NO_START_TEXT / NO_END_TEXT states (so UNCLASSIFIED means “had text but no match”)
  - Respects boolean flags:
      * Start_message == False  -> start labeling is skipped (NO_START_TEXT)
      * End_message   == False  -> end   labeling is skipped (NO_END_TEXT)
    (And these skipped rows are excluded from candidate mining + label distributions.)
  - Margin-based confidence + evidence columns
  - Candidate mining includes unigrams + bigrams (phrases)
  - parse_known_args() so it runs in Jupyter too
"""

from __future__ import annotations

import argparse
import re
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("english")

# ---- Columns expected in the episodes CSV ----
START_FIELDS = ["start_commit_subject", "start_commit_body", "start_pr_title", "start_pr_body", "start_issue_summary"]
END_FIELDS   = ["end_commit_subject", "end_commit_body", "end_pr_title", "end_pr_body", "end_issue_summary"]

# New boolean flags (your updated input)
START_FLAG_COL = "Start_message"
END_FLAG_COL   = "End_message"

# Minimal stop-list for candidate expansion (ONLY used for candidate mining)
STOP = set("""
a an the and or but if then else when while of to for in on at by from as is are was were be been being with without
this that these those it its i you we they he she them his her our your their into over under up down out off via vs
add adds added adding remove removed removing fix fixes fixed fixing update updates updated updating change changes changed changing
refactor refactors refactored refactoring bump bumps bumped merge merges merged revert reverts reverted
""".split())

# Keep important short tokens for mining (otherwise length filters drop them)
IMPORTANT_SHORT = {"ci", "ui", "e2e", "ftl", "avd", "apk", "aab", "gmd"}

def to_bool(v) -> bool:
    """Robustly parse booleans from bool/int/string cells (so 'False' doesn't become True)."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return False
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return bool(int(v))
    s = str(v).strip().lower()
    if s in {"true", "t", "1", "yes", "y"}:
        return True
    if s in {"false", "f", "0", "no", "n", "", "nan", "none"}:
        return False
    # fallback: treat any other non-empty string as True
    return True

# --------- DEFAULT WINDOWS PATHS (EDIT IF NEEDED) ----------
DEFAULT_BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2")
DEFAULT_OUT_DIR  = DEFAULT_BASE_DIR / "Intention_Detection"
DEFAULT_EPISODES_PATH = DEFAULT_BASE_DIR / "All_episodes_with_messages_TrueFalse.csv"

DEFAULT_DICT_XLSX = DEFAULT_OUT_DIR / "dictionary.xlsx"
DEFAULT_DICT_CSV  = DEFAULT_OUT_DIR / "dictionary.csv"
# -----------------------------------------------------------


def normalize_text_for_tokens(text: str) -> str:
    text = re.sub(r"([a-z])([A-Z])", r"\1 \2", text)  # de-camelcase
    text = text.replace("_", " ").replace("-", " ")
    return text.lower()


def tokenize_and_stem(text: str) -> List[str]:
    """
    Tokenizer that keeps digits so tokens like e2e, v2, aab survive.
    Stemming is applied only to purely alphabetic tokens; alphanumeric tokens are kept as-is.
    """
    if not text:
        return []
    text = normalize_text_for_tokens(text)
    raw = re.findall(r"[a-z0-9]+", text)  # includes digits
    out: List[str] = []
    for t in raw:
        if any(ch.isdigit() for ch in t):
            out.append(t)  # keep alphanumeric tokens unstemmed
        else:
            out.append(stemmer.stem(t))
    return out


def combine_fields(row: pd.Series, fields: List[str]) -> str:
    parts: List[str] = []
    for f in fields:
        if f not in row:
            continue
        v = row.get(f, "")
        if pd.isna(v) or v is None:
            continue
        s = str(v).strip()
        if s:
            parts.append(s)
    return "\n".join(parts)


def count_phrase_occurrences(tokens: List[str], phrase_tokens: List[str]) -> int:
    if not phrase_tokens or not tokens:
        return 0
    if len(phrase_tokens) == 1:
        return sum(1 for t in tokens if t == phrase_tokens[0])
    n = len(phrase_tokens)
    cnt = 0
    for i in range(len(tokens) - n + 1):
        if tokens[i : i + n] == phrase_tokens:
            cnt += 1
    return cnt


def _read_dictionary_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dictionary file not found: {path}")

    suf = path.suffix.lower()
    if suf in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if suf == ".csv":
        return pd.read_csv(path)

    raise ValueError(f"Unsupported dictionary format: {path} (use .xlsx/.xls or .csv)")


def load_dictionary_wide(path: Path) -> Tuple[Dict[str, List[Dict]], List[Dict]]:
    """
    Returns:
      dict_terms: label -> list of {keyword, weight, stem_tokens}
      blacklist_terms: list of {keyword, weight, stem_tokens}   (may be empty)
    """
    df = _read_dictionary_table(path)

    # detect blacklist column by name (case-insensitive)
    blacklist_col = None
    for c in df.columns:
        if str(c).strip().lower() == "blacklist":
            blacklist_col = c
            break

    # Identify label columns (exclude 'no', weight columns, and blacklist column)
    labels = []
    for c in df.columns:
        c_str = str(c)
        if c_str.endswith("__weight"):
            continue
        if blacklist_col is not None and c == blacklist_col:
            continue
        if str(c).strip().lower() == "no":
            continue
        labels.append(c)

    if not labels:
        raise ValueError(f"No label columns found in dictionary: {path}")

    dict_terms: Dict[str, List[Dict]] = {label: [] for label in labels}

    def add_items_for_column(series: pd.Series, weights: Optional[pd.Series]) -> List[Dict]:
        items: List[Dict] = []
        for idx, cell in series.items():
            if pd.isna(cell):
                continue
            cell_s = str(cell).strip()
            if not cell_s:
                continue

            for kw in re.split(r"[;,/|]", cell_s):
                kw = kw.strip()
                if not kw:
                    continue

                wt = 1.0
                if weights is not None:
                    wv = weights.iloc[idx]
                    if not pd.isna(wv):
                        try:
                            wt = float(wv)
                        except Exception:
                            wt = 1.0

                stem_tokens = tokenize_and_stem(kw)
                if not stem_tokens:
                    continue

                items.append({"keyword": kw, "weight": wt, "stem_tokens": stem_tokens})

        # de-dup by stem sequence
        seen = set()
        uniq = []
        for it in items:
            k = " ".join(it["stem_tokens"])
            if k in seen:
                continue
            seen.add(k)
            uniq.append(it)
        return uniq

    # load label keywords
    for label in labels:
        wcol = f"{label}__weight"
        weights = df[wcol] if wcol in df.columns else None
        dict_terms[label] = add_items_for_column(df[label], weights) if label in df.columns else []

    # load blacklist keywords (optional)
    blacklist_terms: List[Dict] = []
    if blacklist_col is not None:
        bwcol = f"{blacklist_col}__weight"
        bweights = df[bwcol] if bwcol in df.columns else None
        blacklist_terms = add_items_for_column(df[blacklist_col], bweights)

    return dict_terms, blacklist_terms


def classify_text(
    text: str,
    dict_terms: Dict[str, List[Dict]],
    blacklist_terms: List[Dict],
    cap_per_keyword: bool = True
) -> Tuple[Dict[str, float], Dict[str, List[str]], List[str]]:
    """
    Returns:
      scores: label -> float
      matches: label -> list[keywords matched]
      blacklist_hits: list[keywords matched in blacklist]
    """
    tokens = tokenize_and_stem(text or "")
    scores = {label: 0.0 for label in dict_terms}
    matches = {label: [] for label in dict_terms}

    # Blacklist hits (noise indicators)
    blacklist_hits: List[str] = []
    for it in blacklist_terms:
        occ = count_phrase_occurrences(tokens, it["stem_tokens"])
        if occ > 0:
            blacklist_hits.append(it["keyword"])

    # Label scoring
    for label, items in dict_terms.items():
        for it in items:
            occ = count_phrase_occurrences(tokens, it["stem_tokens"])
            if occ > 0:
                if cap_per_keyword:
                    occ = 1
                scores[label] += it["weight"] * occ
                matches[label].append(it["keyword"])

    return scores, matches, sorted(set(blacklist_hits))


def assign_labels_multi(
    scores: Dict[str, float],
    matches: Dict[str, List[str]],
    label_if_no_text: str,
    text_is_empty: bool,
    blacklist_hits: List[str],
    min_score: float = 1.0,
    multi_ratio: float = 0.8,
    max_labels: int = 3,
) -> Dict[str, object]:
    """
    Returns a dict with:
      label_str
      top_label, top_score, second_label, second_score
      confidence_margin  ( (top-second)/top )
      confidence_share   ( top / sum_scores )
      matched_keywords
      blacklist_hits
    """
    if text_is_empty:
        return {
            "label_str": label_if_no_text,
            "top_label": "", "top_score": 0.0,
            "second_label": "", "second_score": 0.0,
            "confidence_margin": 0.0,
            "confidence_share": 0.0,
            "matched_keywords": "",
            "blacklist_hits": "; ".join(blacklist_hits),
        }

    items = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    top_label, top_score = items[0]
    second_label, second_score = (items[1] if len(items) > 1 else ("", 0.0))

    total = sum(scores.values())
    conf_share = (top_score / total) if total > 0 else 0.0
    conf_margin = ((top_score - second_score) / top_score) if top_score > 0 else 0.0

    # If nothing matches -> UNCLASSIFIED
    if top_score < min_score:
        label = "UNCLASSIFIED_NOISE" if blacklist_hits else "UNCLASSIFIED"
        return {
            "label_str": label,
            "top_label": top_label, "top_score": float(top_score),
            "second_label": second_label, "second_score": float(second_score),
            "confidence_margin": float(conf_margin),
            "confidence_share": float(conf_share),
            "matched_keywords": "",
            "blacklist_hits": "; ".join(blacklist_hits),
        }

    chosen = []
    for lab, sc in items:
        if sc < min_score:
            break
        if sc >= top_score * multi_ratio:
            chosen.append(lab)
        if len(chosen) >= max_labels:
            break

    matched = sorted({kw for lab in chosen for kw in matches.get(lab, [])})

    return {
        "label_str": " || ".join(chosen),
        "top_label": top_label, "top_score": float(top_score),
        "second_label": second_label, "second_score": float(second_score),
        "confidence_margin": float(conf_margin),
        "confidence_share": float(conf_share),
        "matched_keywords": "; ".join(matched),
        "blacklist_hits": "; ".join(blacklist_hits),
    }


def _valid_token_for_mining(t: str) -> bool:
    if t in STOP:
        return False
    if len(t) >= 3:
        return True
    return t in IMPORTANT_SHORT


def ngrams(tokens: List[str], n: int) -> List[str]:
    if n <= 1:
        return tokens[:]
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def build_candidate_suggestions(
    df: pd.DataFrame,
    text_fields: List[str],
    label_col: str,
    dict_terms: Dict[str, List[Dict]],
    blacklist_terms: List[Dict],
    top_n: int = 200,
    min_freq: int = 5,
    ratio: float = 1.5,
    min_assoc: int = 3,
    use_bigrams: bool = True,
) -> pd.DataFrame:
    # existing dictionary entries to exclude (stem sequences)
    dict_stem_seqs = set()
    for items in dict_terms.values():
        for it in items:
            dict_stem_seqs.add(" ".join(it["stem_tokens"]))
    for it in blacklist_terms:
        dict_stem_seqs.add(" ".join(it["stem_tokens"]))

    # candidates from UNCLASSIFIED only (exclude NO_TEXT)
    cand_counts = defaultdict(int)
    uncls = df[df[label_col].astype(str).isin(["UNCLASSIFIED", "UNCLASSIFIED_NOISE"])]

    for _, row in uncls.iterrows():
        text = combine_fields(row, text_fields)
        toks = [t for t in tokenize_and_stem(text) if _valid_token_for_mining(t)]

        for t in toks:
            if t in dict_stem_seqs:
                continue
            cand_counts[t] += 1

        if use_bigrams and len(toks) >= 2:
            for bg in ngrams(toks, 2):
                if bg in dict_stem_seqs:
                    continue
                cand_counts[bg] += 1

    cands = [(t, c) for t, c in cand_counts.items() if c >= min_freq]
    cands.sort(key=lambda x: x[1], reverse=True)
    cands = cands[:top_n]

    # association with already-classified items (primary label)
    labeled = df[~df[label_col].astype(str).isin(["UNCLASSIFIED", "UNCLASSIFIED_NOISE", "NO_START_TEXT", "NO_END_TEXT"])].copy()
    labeled["primary"] = labeled[label_col].astype(str).map(lambda s: (s.split("||")[0].strip() if s else ""))

    labeled_token_sets = []
    labeled_bigram_sets = []
    for _, row in labeled.iterrows():
        text = combine_fields(row, text_fields)
        toks = [t for t in tokenize_and_stem(text) if _valid_token_for_mining(t)]
        labeled_token_sets.append(set(toks))
        labeled_bigram_sets.append(set(ngrams(toks, 2)) if use_bigrams else set())

    labels = list(dict_terms.keys())
    suggestions = []

    for cand, freq in cands:
        counts = {lab: 0 for lab in labels}
        is_bigram = (" " in cand)

        for prim, uni_set, bi_set in zip(labeled["primary"].tolist(), labeled_token_sets, labeled_bigram_sets):
            present = (cand in bi_set) if is_bigram else (cand in uni_set)
            if present and prim in counts:
                counts[prim] += 1

        items = sorted(counts.items(), key=lambda kv: kv[1], reverse=True)
        top_lab, top_c = items[0]
        top2_lab, top2_c = (items[1] if len(items) > 1 else ("", 0))
        max_other = max([c for lab, c in counts.items() if lab != top_lab], default=0)
        max_rest = max([c for lab, c in counts.items() if lab not in {top_lab, top2_lab}], default=0)

        if top_c >= min_assoc and top_c >= ratio * max_other:
            suggestions.append({
                "keyword": cand,
                "suggested_labels": top_lab,
                "suggested_weight": 2,
                "freq_unclassified": freq,
                **{f"count_{lab}": counts[lab] for lab in labels},
            })
            continue

        if top_c >= min_assoc and top2_c >= min_assoc and top2_c >= ratio * max_rest:
            suggestions.append({
                "keyword": cand,
                "suggested_labels": f"{top_lab} || {top2_lab}",
                "suggested_weight": 1,
                "freq_unclassified": freq,
                **{f"count_{lab}": counts[lab] for lab in labels},
            })

    sug_df = pd.DataFrame(suggestions)
    if not sug_df.empty:
        sug_df.sort_values(["suggested_weight", "freq_unclassified"], ascending=[False, False], inplace=True)
    return sug_df


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--episodes", default=str(DEFAULT_EPISODES_PATH), help="Path to episodes CSV")
    ap.add_argument(
        "--dictionary",
        default=str(DEFAULT_DICT_XLSX if DEFAULT_DICT_XLSX.exists() else DEFAULT_DICT_CSV),
        help="Path to dictionary.xlsx or dictionary.csv",
    )
    ap.add_argument("--out_dir", default=str(DEFAULT_OUT_DIR), help="Output directory")

    ap.add_argument("--min_score", type=float, default=1.0, help="Minimum label score to classify")
    ap.add_argument("--multi_ratio", type=float, default=0.8, help="Keep additional labels if score >= top*multi_ratio")
    ap.add_argument("--top_n_candidates", type=int, default=200)
    ap.add_argument("--min_candidate_freq", type=int, default=5)
    ap.add_argument("--use_bigrams", type=int, default=1, help="1=mine bigrams, 0=unigrams only")

    # IMPORTANT: parse_known_args ignores Jupyter's injected args like --f=...kernel.json
    args, _unknown = ap.parse_known_args()

    episodes_path = Path(args.episodes)
    dict_path = Path(args.dictionary)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    if not episodes_path.exists():
        raise FileNotFoundError(f"Episodes CSV not found: {episodes_path}")
    if not dict_path.exists():
        raise FileNotFoundError(f"Dictionary file not found: {dict_path}")

    df = pd.read_csv(episodes_path)
    dict_terms, blacklist_terms = load_dictionary_wide(dict_path)

    # Ensure flags exist; if missing, assume True
    if START_FLAG_COL not in df.columns:
        df[START_FLAG_COL] = True
    if END_FLAG_COL not in df.columns:
        df[END_FLAG_COL] = True

    # --- classify start/end ---
    start_out = []
    end_out = []

    start_used_in_model = []
    end_used_in_model = []

    for _, row in df.iterrows():
        # START
        start_flag = to_bool(row.get(START_FLAG_COL, True))
        stext = combine_fields(row, START_FIELDS).strip() if start_flag else ""
        sc, sm, sblack = classify_text(stext, dict_terms, blacklist_terms) if stext else (
            {k: 0.0 for k in dict_terms}, {k: [] for k in dict_terms}, []
        )
        sres = assign_labels_multi(
            sc, sm,
            label_if_no_text="NO_START_TEXT",
            text_is_empty=(len(stext) == 0),
            blacklist_hits=sblack,
            min_score=args.min_score,
            multi_ratio=args.multi_ratio,
        )
        start_out.append(sres)
        start_used_in_model.append(bool(start_flag) and len(stext) > 0)

        # END
        end_flag = to_bool(row.get(END_FLAG_COL, True))
        etext = combine_fields(row, END_FIELDS).strip() if end_flag else ""
        ec, em, eblack = classify_text(etext, dict_terms, blacklist_terms) if etext else (
            {k: 0.0 for k in dict_terms}, {k: [] for k in dict_terms}, []
        )
        eres = assign_labels_multi(
            ec, em,
            label_if_no_text="NO_END_TEXT",
            text_is_empty=(len(etext) == 0),
            blacklist_hits=eblack,
            min_score=args.min_score,
            multi_ratio=args.multi_ratio,
        )
        end_out.append(eres)
        end_used_in_model.append(bool(end_flag) and len(etext) > 0)

    df["start_used_in_model"] = start_used_in_model
    df["end_used_in_model"] = end_used_in_model

    # attach outputs
    df["start_intent_labels"] = [x["label_str"] for x in start_out]
    df["start_top_label"] = [x["top_label"] for x in start_out]
    df["start_top_score"] = [x["top_score"] for x in start_out]
    df["start_second_label"] = [x["second_label"] for x in start_out]
    df["start_second_score"] = [x["second_score"] for x in start_out]
    df["start_conf_margin"] = [x["confidence_margin"] for x in start_out]
    df["start_conf_share"] = [x["confidence_share"] for x in start_out]
    df["start_intent_keywords"] = [x["matched_keywords"] for x in start_out]
    df["start_blacklist_hits"] = [x["blacklist_hits"] for x in start_out]

    df["end_intent_labels"] = [x["label_str"] for x in end_out]
    df["end_top_label"] = [x["top_label"] for x in end_out]
    df["end_top_score"] = [x["top_score"] for x in end_out]
    df["end_second_label"] = [x["second_label"] for x in end_out]
    df["end_second_score"] = [x["second_score"] for x in end_out]
    df["end_conf_margin"] = [x["confidence_margin"] for x in end_out]
    df["end_conf_share"] = [x["confidence_share"] for x in end_out]
    df["end_intent_keywords"] = [x["matched_keywords"] for x in end_out]
    df["end_blacklist_hits"] = [x["blacklist_hits"] for x in end_out]

    out_eps = out_dir / "episodes_with_intentions.csv"
    df.to_csv(out_eps, index=False, encoding="utf-8")

    # --- candidate suggestions (only from rows that were eligible + had text) ---
    start_df = df[df["start_used_in_model"]].copy()
    end_df = df[df["end_used_in_model"]].copy()

    start_sug = build_candidate_suggestions(
        start_df, START_FIELDS, "start_intent_labels", dict_terms, blacklist_terms,
        top_n=args.top_n_candidates,
        min_freq=args.min_candidate_freq,
        use_bigrams=bool(args.use_bigrams),
    )
    end_sug = build_candidate_suggestions(
        end_df, END_FIELDS, "end_intent_labels", dict_terms, blacklist_terms,
        top_n=args.top_n_candidates,
        min_freq=args.min_candidate_freq,
        use_bigrams=bool(args.use_bigrams),
    )

    start_sug.to_csv(out_dir / "start_intent_candidate_keywords.csv", index=False, encoding="utf-8")
    end_sug.to_csv(out_dir / "end_intent_candidate_keywords.csv", index=False, encoding="utf-8")

    # --- console summary (eligible rows only) ---
    print("[ok] wrote:", out_eps)
    print("[ok] wrote:", out_dir / "start_intent_candidate_keywords.csv")
    print("[ok] wrote:", out_dir / "end_intent_candidate_keywords.csv")
    print("\nStart label distribution (eligible rows only):")
    print(start_df["start_intent_labels"].value_counts(dropna=False).head(25))
    print("\nEnd label distribution (eligible rows only):")
    print(end_df["end_intent_labels"].value_counts(dropna=False).head(25))
    print("\n[info] out_dir:", out_dir)


if __name__ == "__main__":
    main()

[ok] wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\Intention_Detection\episodes_with_intentions.csv
[ok] wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\Intention_Detection\start_intent_candidate_keywords.csv
[ok] wrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3_2\Intention_Detection\end_intent_candidate_keywords.csv

Start label distribution:
start_intent_labels
Migrate_or_modernise_CI_infrastructure                                                                                       111
UNCLASSIFIED                                                                                                                  89
UNCLASSIFIED_NOISE                                                                                                            47
Expand_test_scope_or_capabilities                                                                                      